# CIFAR-10 Image Classification — Week 4 Assignment (Intro to DL)

Covers: Perceptron/MLP/backprop, activation functions, CNN (conv/pooling/stride/padding), training strategies (EarlyStopping, augmentation), and transfer learning.

Models built: baseline ANN, baseline CNN, activation comparison (sigmoid vs ReLU), deeper ANN, deeper CNN, CNN + augmentation, transfer learning (MobileNetV2).
All models are compared on test accuracy at the end.

## Setup

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices('GPU')
print("GPUs found:", gpus)
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)  # avoids OOM crashes on WSL/local GPUs

## Load CIFAR-10

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print("Train shape:", x_train.shape, "Test shape:", x_test.shape)

### Class distribution and sample images

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# class balance check
counts = pd.Series(y_train.flatten()).map(lambda i: class_names[i]).value_counts().sort_index()
axes[0].bar(counts.index, counts.values)
axes[0].set_title("Training samples per class")
axes[0].tick_params(axis='x', rotation=45)

axes[1].axis("off")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis("off")
plt.tight_layout()
plt.show()

## Preprocessing
Normalize pixels to 0-1. ANN needs flattened vectors, CNN keeps the 32x32x3 shape.

In [ ]:
x_train_norm = x_train / 255.0
x_test_norm = x_test / 255.0

x_train_flat = x_train_norm.reshape(len(x_train_norm), -1)
x_test_flat = x_test_norm.reshape(len(x_test_norm), -1)

y_train = y_train.flatten()
y_test = y_test.flatten()

print("Flattened ANN input shape:", x_train_flat.shape)

## Part 1: Baseline ANN
A plain MLP (perceptron layers stacked + backprop) — no convolution, so it treats the image as a flat vector and loses spatial structure. This is the "before" reference.

In [ ]:
ann_baseline = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
])

ann_baseline.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

ann_baseline.summary()

ann_baseline_history = ann_baseline.fit(
    x_train_flat, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64
)

In [ ]:
ann_baseline_loss, ann_baseline_acc = ann_baseline.evaluate(x_test_flat, y_test, verbose=0)
print("Baseline ANN test accuracy:", ann_baseline_acc)

## Activation function experiment: Sigmoid vs ReLU
Same architecture, only the activation changes. Illustrates why ReLU trains faster and avoids vanishing gradients compared to sigmoid — trained on a 10k subset so it runs quickly.

In [ ]:
def build_activation_ann(activation):
    model = models.Sequential([
        layers.Dense(256, activation=activation, input_shape=(3072,)),
        layers.Dense(128, activation=activation),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

x_sub, y_sub = x_train_flat[:10000], y_train[:10000]

sigmoid_model = build_activation_ann('sigmoid')
sigmoid_history = sigmoid_model.fit(x_sub, y_sub, epochs=10, validation_split=0.1,
                                     batch_size=64, verbose=0)

relu_model = build_activation_ann('relu')
relu_history = relu_model.fit(x_sub, y_sub, epochs=10, validation_split=0.1,
                               batch_size=64, verbose=0)

plt.figure(figsize=(10, 4))
plt.plot(sigmoid_history.history['loss'], label='Sigmoid train loss')
plt.plot(relu_history.history['loss'], label='ReLU train loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Sigmoid vs ReLU convergence speed (10k samples)")
plt.legend()
plt.show()

## Task 1: Deeper ANN
More layers + BatchNorm + EarlyStopping, 20 epochs. EarlyStopping restores the best weights so extra epochs never hurt — it just stops once val_loss stops improving.

In [ ]:
ann_deep = models.Sequential([
    layers.Dense(1024, activation='relu', input_shape=(3072,)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

ann_deep.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

ann_deep_history = ann_deep.fit(
    x_train_flat, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop]
)

In [ ]:
ann_deep_loss, ann_deep_acc = ann_deep.evaluate(x_test_flat, y_test, verbose=0)
print("Deep ANN test accuracy:", ann_deep_acc)

## Part 2: Baseline CNN
Convolution preserves spatial structure. `padding='same'` keeps the output size equal to the input size per conv layer; `MaxPooling2D` then halves it (stride 2) to build up receptive field.

In [ ]:
cnn_baseline = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

cnn_baseline.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

cnn_baseline.summary()

cnn_baseline_history = cnn_baseline.fit(
    x_train_norm, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64
)

In [ ]:
cnn_baseline_loss, cnn_baseline_acc = cnn_baseline.evaluate(x_test_norm, y_test, verbose=0)
print("Baseline CNN test accuracy:", cnn_baseline_acc)

## Task 2: Deeper CNN (more filters, doubled conv blocks)
Filter progression 32 -> 64 -> 128, with two conv layers per block before pooling. EarlyStopping + 20 epochs again.

In [ ]:
cnn_deep = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

cnn_deep.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

cnn_deep_history = cnn_deep.fit(
    x_train_norm, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]
)

In [ ]:
cnn_deep_loss, cnn_deep_acc = cnn_deep.evaluate(x_test_norm, y_test, verbose=0)
print("Deep CNN test accuracy:", cnn_deep_acc)

## Task 5: CNN + Data Augmentation (actually trained)
RandomFlip/Rotation/Zoom generate slightly different images each epoch, which reduces overfitting. Uses the same deep CNN architecture as above so the comparison is apples-to-apples.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

def build_cnn_backbone():
    return models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])

cnn_aug = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    build_cnn_backbone()
])

cnn_aug.compile(optimizer='adam',
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

cnn_aug_history = cnn_aug.fit(
    x_train_norm, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]
)

In [ ]:
cnn_aug_loss, cnn_aug_acc = cnn_aug.evaluate(x_test_norm, y_test, verbose=0)
print("Augmented CNN test accuracy:", cnn_aug_acc)

## Transfer Learning: MobileNetV2
CIFAR-10 images are 32x32; MobileNetV2 expects larger inputs, so they're resized to 96x96 on the fly via `tf.data` (avoids holding a resized copy of the whole dataset in memory). The pretrained base is frozen first, then the top layers are unfrozen for fine-tuning at a low learning rate.

In [ ]:
IMG_SIZE = 96
BATCH_SIZE = 64

# hold out a validation slice from the training set
val_frac = 0.1
n_val = int(len(x_train) * val_frac)
x_tr, y_tr = x_train[n_val:], y_train[n_val:]
x_val, y_val = x_train[:n_val], y_train[:n_val]

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label

def make_ds(x, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(10000, seed=SEED)
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_ds(x_tr, y_tr, training=True)
val_ds = make_ds(x_val, y_val)
test_ds = make_ds(x_test, y_test)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

tl_model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

tl_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

tl_history_head = tl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)]
)

### Fine-tune: unfreeze the top of the base model at a low learning rate

In [ ]:
base_model.trainable = True
# freeze everything except the last ~20 layers
for layer in base_model.layers[:-20]:
    layer.trainable = False

tl_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

tl_history_finetune = tl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)]
)

In [ ]:
tl_loss, tl_acc = tl_model.evaluate(test_ds, verbose=0)
print("Transfer learning (MobileNetV2) test accuracy:", tl_acc)

## Compare all models

In [ ]:
results = pd.DataFrame({
    "Model": ["ANN (baseline)", "ANN (deep + EarlyStopping)",
              "CNN (baseline)", "CNN (deep + EarlyStopping)",
              "CNN (deep + augmentation)", "Transfer learning (MobileNetV2)"],
    "Test Accuracy": [ann_baseline_acc, ann_deep_acc,
                       cnn_baseline_acc, cnn_deep_acc,
                       cnn_aug_acc, tl_acc],
    "Test Loss": [ann_baseline_loss, ann_deep_loss,
                   cnn_baseline_loss, cnn_deep_loss,
                   cnn_aug_loss, tl_loss]
}).sort_values("Test Accuracy", ascending=False).reset_index(drop=True)

results

In [ ]:
plt.figure(figsize=(10, 5))
plt.barh(results["Model"], results["Test Accuracy"])
plt.xlabel("Test Accuracy")
plt.title("Model comparison — CIFAR-10 test accuracy")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Learning curves: train vs val, accuracy and loss

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

histories = {
    "ANN (deep)": ann_deep_history,
    "CNN (deep)": cnn_deep_history,
    "CNN (augmented)": cnn_aug_history,
}

for name, h in histories.items():
    axes[0, 0].plot(h.history['accuracy'], label=name)
    axes[0, 1].plot(h.history['val_accuracy'], label=name)
    axes[1, 0].plot(h.history['loss'], label=name)
    axes[1, 1].plot(h.history['val_loss'], label=name)

axes[0, 0].set_title("Train accuracy")
axes[0, 1].set_title("Val accuracy")
axes[1, 0].set_title("Train loss")
axes[1, 1].set_title("Val loss")
for ax in axes.flat:
    ax.set_xlabel("Epoch")
    ax.legend()

plt.tight_layout()
plt.show()

## Confusion matrix and classification report — best model

In [ ]:
best_model_name = results.iloc[0]["Model"]
print("Best model:", best_model_name)

# map best model name back to its predictions
if best_model_name == "Transfer learning (MobileNetV2)":
    y_pred = np.argmax(tl_model.predict(test_ds, verbose=0), axis=1)
elif best_model_name == "CNN (deep + augmentation)":
    y_pred = np.argmax(cnn_aug.predict(x_test_norm, verbose=0), axis=1)
elif best_model_name == "CNN (deep + EarlyStopping)":
    y_pred = np.argmax(cnn_deep.predict(x_test_norm, verbose=0), axis=1)
elif best_model_name == "CNN (baseline)":
    y_pred = np.argmax(cnn_baseline.predict(x_test_norm, verbose=0), axis=1)
elif best_model_name == "ANN (deep + EarlyStopping)":
    y_pred = np.argmax(ann_deep.predict(x_test_flat, verbose=0), axis=1)
else:
    y_pred = np.argmax(ann_baseline.predict(x_test_flat, verbose=0), axis=1)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 7))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xticks(range(10), class_names, rotation=45)
plt.yticks(range(10), class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Confusion matrix — {best_model_name}")
for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j], ha='center', va='center',
                  color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=8)
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=class_names))

### A few misclassified examples

In [ ]:
wrong_idx = np.where(y_pred != y_test)[0]
sample_idx = np.random.choice(wrong_idx, size=min(10, len(wrong_idx)), replace=False)

plt.figure(figsize=(12, 5))
for i, idx in enumerate(sample_idx):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[idx])
    plt.title(f"True: {class_names[y_test[idx]]}\nPred: {class_names[y_pred[idx]]}", fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

## Conclusion

- ANN treats images as flat vectors and ignores spatial structure — CNN consistently outperforms it.
- ReLU converges faster than sigmoid on this data, consistent with the vanishing-gradient issue sigmoid has in deeper networks.
- Adding depth/filters to the CNN improves capacity, but EarlyStopping is what keeps 20-epoch runs from overfitting.
- Data augmentation narrows the train/val gap, i.e. reduces overfitting, though it doesn't always beat a well-regularized plain CNN outright.
- Transfer learning (MobileNetV2 pretrained on ImageNet) starts from features already useful for natural images, so it can reach strong accuracy with comparatively little training — this is the main practical benefit of transfer learning over training from scratch.